In [118]:
import pandas as pd
import json
import os


In [119]:
visus_folder_name = 'visualizations'
os.makedirs(visus_folder_name, exist_ok=True)
visus_path = os.path.join(os.getcwd(), visus_folder_name)

In [120]:
base_path = "evaluation"
cenarios = ["zero-shot", "few-shot"]
dados = []

for cenario in cenarios:
    cenario_path = os.path.join(base_path, cenario)
    
    for root, dirs, files in os.walk(cenario_path):
        for file in files:
            if file == "individual_metrics.json":
                file_path = os.path.join(root, file)
                
                with open(file_path, "r") as f:
                    metrics = json.load(f)
                
                for modelo, valores in metrics.items():
                    dados.append({
                        "cenario": cenario,
                        "modelo": modelo,
                        **valores
                    })

df_individuals = pd.DataFrame(dados)
df_individuals.head()

,cenario,modelo,punchlines,comic_styles,texts_explanations
0,zero-shot,gemma-3-gaia-pt-br-4b-it,[{'video_url': 'https://www.youtube.com/shorts...,[{'video_url': 'https://www.youtube.com/shorts...,[{'video_url': 'https://www.youtube.com/shorts...
1,zero-shot,mixtral-8x7b-instruct-v01,[{'video_url': 'https://www.youtube.com/shorts...,[{'video_url': 'https://www.youtube.com/shorts...,[{'video_url': 'https://www.youtube.com/shorts...
2,zero-shot,sabia-3.1,[{'video_url': 'https://www.youtube.com/shorts...,[{'video_url': 'https://www.youtube.com/shorts...,[{'video_url': 'https://www.youtube.com/shorts...
3,zero-shot,granite-3-3-8b-instruct,[{'video_url': 'https://www.youtube.com/shorts...,[{'video_url': 'https://www.youtube.com/shorts...,[{'video_url': 'https://www.youtube.com/shorts...
4,zero-shot,gemini-2.5-flash,[{'video_url': 'https://www.youtube.com/shorts...,[{'video_url': 'https://www.youtube.com/shorts...,[{'video_url': 'https://www.youtube.com/shorts...


In [121]:
def expand_dict_column(df, dict_col):
    df = df.copy()
    df[dict_col] = df[dict_col].apply(
        lambda x: x if isinstance(x, list) else [x] if isinstance(x, dict) else []
    )

    expanded_rows = []
    for _, row in df.iterrows():
        for item in row[dict_col]:
            if isinstance(item, dict):
                expanded_rows.append({
                    "scenario": row["cenario"],
                    **item
                })

    return pd.DataFrame(expanded_rows)

df_punchlines = expand_dict_column(df_individuals, "punchlines")
df_comic_styles = expand_dict_column(df_individuals, "comic_styles")
df_texts_explanations = expand_dict_column(df_individuals, "texts_explanations")
df_texts_explanations['agreement_level'] = df_texts_explanations['judge_model_results'].apply(lambda judge_model_result: int(judge_model_result['nivel_concordancia']))

### Erros e acertos em comum 

In [122]:
MIN_MODELS_REPEATS = 5

WORSE_PUNCHLINES_THRESHOLD = 0.4
BETTER_PUNCHLINES_THRESHOLD = 0.6

WORSE_EXPLANATIONS_THRESHOLD = 2
BETTER_EXPLANATIONS_THRESHOLD = 4

##### Punchlines Identification

In [123]:
df_punchlines_worse = df_punchlines[df_punchlines['dice_similarity'] <= WORSE_PUNCHLINES_THRESHOLD]
df_punchlines_better = df_punchlines[df_punchlines['dice_similarity'] >= BETTER_PUNCHLINES_THRESHOLD]

##### Humor Reasoning

In [124]:
df_explanations_worse = df_texts_explanations[df_texts_explanations['agreement_level'] <= WORSE_EXPLANATIONS_THRESHOLD]
df_explanations_better = df_texts_explanations[df_texts_explanations['agreement_level'] <= BETTER_EXPLANATIONS_THRESHOLD]

##### Comic Styles Classification

In [125]:
comic_styles = df_comic_styles['comic_style'].unique()

print(comic_styles)

# O código abaixo cria dois dataframe de para cada estilo cômico, um de piores casos e outro de melhores. 
# Ex.: df_fun_worse, df_fun_better, df_irony_worse, df_irony_better.

for style in comic_styles:
    df_name_worse = f"df_{style}_worse"
    df_name_better = f"df_{style}_better"
    globals()[df_name_worse] = df_comic_styles[(df_comic_styles['comic_style'] == style) & (df_comic_styles['is_correct'] == 0)]
    globals()[df_name_better] = df_comic_styles[(df_comic_styles['comic_style'] == style) & (df_comic_styles['is_correct'] == 1)]

['fun' 'humor' 'nonsense' 'wit' 'irony' 'satire' 'sarcasm' 'cynicism']


In [126]:
df_irony_worse.head(3)

,scenario,video_url,comic_style,prompt,true_label,pred_label,is_correct,humorous_text,model_name
12,zero-shot,https://www.youtube.com/shorts/QV80EKixLYc?fea...,irony,"Dado o seguinte texto humorístico, avalie se e...",0,1,0,Humorista: Quem já passou algumas coisas muito...,gemma-3-gaia-pt-br-4b-it
20,zero-shot,https://www.youtube.com/shorts/yn3-Kmp7hjM?fea...,irony,"Dado o seguinte texto humorístico, avalie se e...",0,1,0,"O calor é estranho, né? Porque é um calor de c...",gemma-3-gaia-pt-br-4b-it
52,zero-shot,https://www.youtube.com/shorts/i0XFdPvJC_E?fea...,irony,"Dado o seguinte texto humorístico, avalie se e...",0,1,0,A gente se conheceu num bar na frente da escol...,gemma-3-gaia-pt-br-4b-it
